# PROYEK AKHIR SEMESTER — PBO 2026

## Analisis Data XAU/USD (Gold) menggunakan *Object-Oriented Programming*

**Dataset:** XAU/USD Gold Price Historical Data 2004–2024 (Kaggle / novandraanugrah)
**Bahasa:** Python 3.10+
**Pendekatan:** Object-Oriented Programming dengan arsitektur berlapis (5-tier) dan Repository Pattern

---

| | |
|---|---|
| **Nama Mahasiswa** | *[Nama Lengkap]* |
| **NIM** | *[NIM]* |
| **Mata Kuliah** | Pemrograman Berorientasi Objek (PBO) 2026 |
| **Program Studi** | Informatika |
| **Universitas** | Universitas Mulawarman |


## Daftar Isi

1. Latar Belakang & Rumusan Masalah
2. Tujuan & Manfaat Proyek
3. Deskripsi Dataset EUR/USD
4. Landasan Teori OOP
5. Desain Sistem & Arsitektur
6. Class Diagram (UML)
7. Implementasi Kode
    1. Model Layer — `Candle`
    2. Repository Layer — `CandleRepository`
    3. Analyzer Layer — `BaseAnalyzer` + 6 subclass
    4. Service Layer — `ForexAnalysisService`
8. Eksekusi Pipeline Analisis
9. Hasil Analisis Numerik
10. Visualisasi & Interpretasi
11. Pembahasan
12. Kesimpulan & Saran
13. Daftar Pustaka


## 1. Latar Belakang & Rumusan Masalah

### Latar Belakang

- **Emas (XAU/USD)** adalah salah satu instrumen finansial paling penting di dunia, berperan sebagai *safe-haven asset* yang dicari investor saat ketidakpastian ekonomi.
- Harga emas dipengaruhi banyak faktor: kebijakan moneter bank sentral, inflasi, geopolitik, dan permintaan industri & perhiasan.
- Periode 2004–2024 (~21 tahun) mencakup beberapa peristiwa besar: krisis keuangan 2008, periode QE, COVID-19, inflasi pasca-pandemi 2022–2023, dan reli emas 2024.
- Pendekatan **Object-Oriented Programming (OOP)** memungkinkan pembangunan sistem analisis yang **modular, reusable, dan dapat dikembangkan** sesuai prinsip rekayasa perangkat lunak modern.
- Penulis memiliki pengalaman trading pribadi pada pasangan XAU/USD menggunakan analisis teknikal di timeframe 1 menit, sehingga analisis ini juga berfungsi sebagai studi lebih dalam pada timeframe yang lebih lama (daily) dengan pendekatan akademis.

### Rumusan Masalah

1. Bagaimana karakteristik pergerakan harga XAU/USD selama periode 2004–2024 (trend jangka panjang, volatilitas, distribusi returns)?
2. Bagaimana merancang sistem analisis time-series menggunakan OOP yang memenuhi prinsip SOLID dan menerapkan Repository Pattern?
3. Bagaimana menerapkan keseluruhan pilar OOP (enkapsulasi, pewarisan, polimorfisme, abstraksi) pada pipeline analisis data komoditas?
4. Insight kuantitatif apa yang dapat diturunkan dari analisis tersebut, dan bagaimana cara memvisualisasikannya secara informatif?


## 2. Tujuan & Manfaat

### Tujuan

- Menerapkan seluruh konsep OOP (Materi 01–14) pada proyek analisis data nyata.
- Membangun pipeline analisis time-series yang modular dengan arsitektur 5-layer dan Repository Pattern.
- Mengidentifikasi karakteristik statistik dan pola pergerakan XAU/USD periode 2004–2024.
- Menghasilkan visualisasi yang dapat dibaca dan diinterpretasikan untuk mendukung pengambilan keputusan investasi/trading.

### Manfaat

- **Bagi Penulis:** portfolio proyek OOP end-to-end yang relevan dengan pengalaman trading pribadi pada instrumen yang sama.
- **Bagi Akademik:** referensi penerapan OOP pada domain time-series finansial komoditas.
- **Bagi Pembaca:** template arsitektur yang dapat diadaptasi ke instrumen lain (XAG/USD perak, pasangan mata uang, indeks saham, kripto).


## 3. Deskripsi Dataset XAU/USD

**Sumber:** Kaggle — [XAU/USD Gold Price Historical Data 2004–2024](https://www.kaggle.com/datasets/novandraanugrah/xauusd-gold-price-historical-data-2004-2024/data?select=XAU_1d_data.csv)

| Atribut | Nilai |
|---|---|
| Nama file | `XAU_1d_data.csv` |
| Periode | 2004 – 2024 (~21 tahun) |
| Timeframe | **Harian (1D)** — sudah daily, tidak perlu resampling |
| Tipe data | CSV |
| Variabel | OHLCV (Open, High, Low, Close, Volume) |

### Karakter Aset XAU/USD

Berbeda dengan pasangan mata uang (forex pair), XAU/USD memiliki karakter khas:

- **Volatilitas lebih tinggi** dibanding major forex pair (range harian 1–3% normal, bisa 5%+ saat event ekstrem).
- **Trending lebih kuat**, terutama pada periode krisis (2008, 2020) — sifat *safe-haven* tercermin di lonjakan harga.
- **Korelasi negatif dengan USD index (DXY)** dan **positif dengan tingkat inflasi**.
- **Skala harga besar** — beberapa ratus hingga ribuan USD per troy ounce (sekitar $400 di 2004 hingga $2,700+ di 2024).

### Pembersihan & Validasi Data

`CandleRepository` melakukan validasi data saat loading:
- Parsing tanggal ke `datetime` Python.
- Validasi setiap row sebagai instance `Candle` (harga > 0, high ≥ low, open/close dalam range [low, high]).
- Sortir berdasarkan tanggal (ascending).


## 4. Landasan Teori — Konsep OOP yang Diterapkan

Proyek ini menerapkan seluruh pilar OOP yang dipelajari di Materi 01–14:

| Pilar / Materi | Implementasi di Proyek |
|---|---|
| **Enkapsulasi** (Materi 4) | Atribut protected `_data`, `_candles`, `_lookback` di Analyzer & Repository |
| **Pewarisan** (Materi 6) | 6 subclass Analyzer mewarisi `BaseAnalyzer` |
| **Polimorfisme** (Materi 7) | Method `analyze()` di-override dengan implementasi berbeda per subclass |
| **Abstraksi** (Materi 8) | `BaseAnalyzer(ABC)` + `@abstractmethod` memaksa subclass mengimplementasikan `analyze()` |
| **Magic Methods** (Materi 9) | `Candle.__str__`, `Candle.__lt__`, `Candle.__post_init__`, `Repository.__len__` |
| **Exception Handling** (Materi 10) | Hierarki custom: `ForexDataError` → `InvalidCandleError`, `RepositoryNotLoadedError`, `AnalysisError` |
| **SOLID** (Materi 11) | SRP per class, OCP via ABC (tambah analyzer = tambah subclass), DIP (Service depends on abstraksi) |
| **Design Pattern** (Materi 12) | **Repository Pattern** (memisahkan akses data), **Strategy Pattern** (Analyzer sebagai strategi analisis) |
| **OOP Modern** (Materi 13) | `@dataclass(frozen=True)` pada `Candle`, type hints, immutability |
| **File & DB** (Materi 14) | Load CSV dengan `pandas`, export report ke JSON via `json.dump` |


## 5. Desain Sistem & Arsitektur

Arsitektur **5-Layer** dengan Repository Pattern, mengikuti prinsip Separation of Concerns:

```
+------------------------------------------------+
|  MAIN / NOTEBOOK EXECUTION CELLS                |
|  Sel-sel di bagian section 8 - orkestrasi       |
+--------------------+---------------------------+
                     v
+------------------------------------------------+
|  SERVICE LAYER                                  |
|  ForexAnalysisService                           |
|  Mengorkestrasi semua analyzer, hasilkan report |
+--------------------+---------------------------+
                     v
+------------------------------------------------+
|  ANALYZER LAYER (Strategy Pattern via ABC)      |
|  BaseAnalyzer (abstract)                        |
|  +-- TrendAnalyzer                              |
|  +-- VolatilityAnalyzer                         |
|  +-- ReturnsAnalyzer                            |
|  +-- MovingAverageAnalyzer                      |
|  +-- PriceDistributionAnalyzer                  |
|  +-- LiquiditySwingAnalyzer                     |
+--------------------+---------------------------+
                     v
+------------------------------------------------+
|  REPOSITORY LAYER                               |
|  CandleRepository                               |
|  Load CSV, parse tanggal, validasi, query       |
+--------------------+---------------------------+
                     v
+------------------------------------------------+
|  MODEL LAYER                                    |
|  Candle (@dataclass(frozen=True))               |
|  + Custom Exceptions                            |
+--------------------+---------------------------+
                     v
+------------------------------------------------+
|  STORAGE                                        |
|  data/XAU_1d_data.csv                           |
+------------------------------------------------+
```


## 6. Class Diagram (UML)

Diagram ini menggambarkan struktur class dan relasinya.

> *(Diagram visual akan disisipkan sebagai gambar dari `outputs/charts/uml_class_diagram.png` pada fase visualisasi. Saat ini ditampilkan dalam bentuk tekstual.)*

```
                       +------------------------------+
                       |  <<dataclass(frozen=True)>>  |
                       |          Candle              |
                       +------------------------------+
                       | + date: datetime             |
                       | + open: float                |
                       | + high: float                |
                       | + low: float                 |
                       | + close: float               |
                       | + volume: float              |
                       +------------------------------+
                       | + __post_init__()            |
                       | + body_size() -> float       |
                       | + range_size() -> float      |
                       | + is_bullish() -> bool       |
                       | + daily_return_pct() -> float|
                       | + __str__(), __lt__()        |
                       +------------------------------+

   CandleRepository (uses Candle)         BaseAnalyzer (abstract, uses Candle)
   - _candles: list                       # _data: list[Candle]
   + load_csv() -> int                    + @abstractmethod analyze()
   + get_all(), get_range(), __len__()    + get_summary() -> dict

   Subclass BaseAnalyzer:
     - TrendAnalyzer
     - VolatilityAnalyzer
     - ReturnsAnalyzer
     - MovingAverageAnalyzer
     - PriceDistributionAnalyzer
     - LiquiditySwingAnalyzer  (terinspirasi LuxAlgo)

   ForexAnalysisService
     <> CandleRepository (aggregation)
     <> list[BaseAnalyzer] (composition)
     + run_full_analysis() -> dict
     + export_report_json(path)
```

### Hierarki Custom Exception

```
Exception (built-in)
    +-- ForexDataError
            +-- InvalidCandleError
            +-- RepositoryNotLoadedError
            +-- AnalysisError
```


## 7. Implementasi Kode

Pada bagian ini seluruh class dari arsitektur 5-layer didefinisikan, dari yang paling dasar (`Candle`) hingga paling atas (`ForexAnalysisService`). Setiap class disertai **inline `assert` tests** untuk memvalidasi behavior pada saat eksekusi notebook.


In [7]:
from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime, date
from pathlib import Path
import json
import logging

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

#logging sederhana
logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
logger = logging.getLogger('forex_analysis')

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print('Library berhasil diimpor.')

Library berhasil diimpor.


### 7.1 Model Layer — `Candle` & Custom Exceptions

`Candle` merepresentasikan satu candle harga (OHLCV). Menggunakan `@dataclass(frozen=True)` agar:
- Boilerplate `__init__`, `__repr__`, `__eq__` auto-generated.
- **Immutable** — instance tidak bisa dimodifikasi setelah dibuat (fail-fast, lebih aman).
- Hashable — bisa dipakai sebagai key dictionary atau anggota set.

Validasi dilakukan di `__post_init__`: harga harus positif, `high >= low`, dst.


In [8]:
# Sel 7.1.1 - Custom Exception Hierarchy
# Materi 10: OOP Exception Handling
# Semua exception domain forex mewarisi dari ForexDataError supaya bisa
# di-catch dengan satu except blok kalau perlu.

class ForexDataError(Exception):
    """Base exception untuk semua error domain forex analysis."""
    pass


class InvalidCandleError(ForexDataError):
    """Raised saat data Candle tidak valid (harga <= 0, high < low, dll)."""
    pass


class RepositoryNotLoadedError(ForexDataError):
    """Raised saat Repository diakses sebelum load_csv() dipanggil."""
    pass


class AnalysisError(ForexDataError):
    """Raised saat analyzer gagal (data kosong, kondisi tidak terpenuhi, dll)."""
    pass


# Verifikasi hierarki exception
assert issubclass(InvalidCandleError, ForexDataError)
assert issubclass(RepositoryNotLoadedError, ForexDataError)
assert issubclass(AnalysisError, ForexDataError)
assert issubclass(ForexDataError, Exception)

print("Custom Exception Hierarchy berhasil didefinisikan:")
print("  Exception (built-in)")
print("    +-- ForexDataError")
print("          +-- InvalidCandleError")
print("          +-- RepositoryNotLoadedError")
print("          +-- AnalysisError")

Custom Exception Hierarchy berhasil didefinisikan:
  Exception (built-in)
    +-- ForexDataError
          +-- InvalidCandleError
          +-- RepositoryNotLoadedError
          +-- AnalysisError


In [9]:
# Sel 7.1.2 - Candle (@dataclass(frozen=True))
# Materi 13: OOP Modern Python (@dataclass, type hints, immutability)
# Materi 04: Enkapsulasi (validasi di __post_init__)
# Materi 09: Magic Methods (__str__, __lt__, __post_init__)

@dataclass(frozen=True)
class Candle:
    """Merepresentasikan satu candle harga OHLCV pada timeframe daily.

    Menggunakan @dataclass(frozen=True) sehingga:
      - Boilerplate __init__, __repr__, __eq__ auto-generated.
      - Instance IMMUTABLE (tidak bisa dimodifikasi setelah dibuat).
      - HASHABLE (bisa dipakai sebagai anggota set atau key dict).

    Validasi dilakukan di __post_init__ - menerapkan prinsip FAIL-FAST.
    """
    date: datetime
    open: float
    high: float
    low: float
    close: float
    volume: float

    # ----- Validasi (__post_init__) -----
    def __post_init__(self) -> None:
        """Validasi konsistensi OHLC. Raise InvalidCandleError kalau gagal."""
        # pd.Timestamp adalah subclass datetime, jadi isinstance check ini OK
        if not isinstance(self.date, datetime):
            raise InvalidCandleError(
                f"date harus datetime, dapat {type(self.date).__name__}"
            )
        # Harga OHLC harus positif (kurs forex tidak pernah <= 0)
        for nm in ("open", "high", "low", "close"):
            v = getattr(self, nm)
            if v <= 0:
                raise InvalidCandleError(f"{nm} harus > 0, dapat {v}")
        # Konsistensi: high >= low
        if self.high < self.low:
            raise InvalidCandleError(
                f"high ({self.high}) harus >= low ({self.low})"
            )
        # Konsistensi: open & close harus di dalam [low, high]
        if not (self.low <= self.open <= self.high):
            raise InvalidCandleError(
                f"open ({self.open}) harus dalam [low={self.low}, high={self.high}]"
            )
        if not (self.low <= self.close <= self.high):
            raise InvalidCandleError(
                f"close ({self.close}) harus dalam [low={self.low}, high={self.high}]"
            )
        # Volume non-negatif
        if self.volume < 0:
            raise InvalidCandleError(f"volume harus >= 0, dapat {self.volume}")

    # ----- Helper methods (Materi 03: instance methods) -----
    def body_size(self) -> float:
        """Ukuran body candle = |close - open|."""
        return abs(self.close - self.open)

    def range_size(self) -> float:
        """Range total candle = high - low (volatilitas intra-day)."""
        return self.high - self.low

    def is_bullish(self) -> bool:
        """True kalau candle bullish (close > open)."""
        return self.close > self.open

    def daily_return_pct(self) -> float:
        """Return harian dalam persen = (close - open) / open * 100."""
        return (self.close - self.open) / self.open * 100.0

    # ----- Magic methods (Materi 09) -----
    def __str__(self) -> str:
        """Representasi string ringkas untuk display & debugging."""
        tag = "BULL" if self.is_bullish() else "BEAR"
        return (
            f"Candle({self.date.strftime('%Y-%m-%d')} [{tag}] "
            f"O={self.open:.5f} H={self.high:.5f} "
            f"L={self.low:.5f} C={self.close:.5f})"
        )

    def __lt__(self, other: "Candle") -> bool:
        """Bandingkan candle berdasarkan tanggal (untuk sorting)."""
        if not isinstance(other, Candle):
            return NotImplemented
        return self.date < other.date


print("Class Candle berhasil didefinisikan (@dataclass(frozen=True)).")
print(f"  Field: date, open, high, low, close, volume")
print(f"  Methods: body_size, range_size, is_bullish, daily_return_pct")
print(f"  Magic: __post_init__, __str__, __lt__")

Class Candle berhasil didefinisikan (@dataclass(frozen=True)).
  Field: date, open, high, low, close, volume
  Methods: body_size, range_size, is_bullish, daily_return_pct
  Magic: __post_init__, __str__, __lt__


In [10]:
# Sel 7.1.3 - Inline tests untuk Candle (9 assertion)
# Materi 10: validasi exception behavior

# --- Test 1: Bullish candle valid ---
c_bull = Candle(
    date=datetime(2021, 1, 4),
    open=1.22340, high=1.23115, low=1.22240, close=1.22990,
    volume=125000.0,
)
assert c_bull.is_bullish() is True
assert abs(c_bull.body_size() - (1.22990 - 1.22340)) < 1e-9
assert abs(c_bull.range_size() - (1.23115 - 1.22240)) < 1e-9
assert c_bull.daily_return_pct() > 0

# --- Test 2: Bearish candle valid ---
c_bear = Candle(
    date=datetime(2021, 1, 5),
    open=1.23000, high=1.23100, low=1.22000, close=1.22500,
    volume=98000.0,
)
assert c_bear.is_bullish() is False
assert c_bear.daily_return_pct() < 0

# --- Test 3: Sorting pakai __lt__ ---
candles = [c_bear, c_bull]
candles_sorted = sorted(candles)
assert candles_sorted[0] is c_bull   # c_bull lebih awal (4 Jan)
assert candles_sorted[-1] is c_bear  # c_bear lebih belakang (5 Jan)

# --- Test 4: Validasi - harga negatif ditolak ---
try:
    Candle(date=datetime(2021, 1, 6), open=-1.0, high=1.5,
           low=0.5, close=1.0, volume=100.0)
    raise AssertionError("Seharusnya raise InvalidCandleError")
except InvalidCandleError:
    pass

# --- Test 5: Validasi - high < low ditolak ---
try:
    Candle(date=datetime(2021, 1, 7), open=1.0, high=0.5,
           low=1.5, close=1.0, volume=100.0)
    raise AssertionError("Seharusnya raise InvalidCandleError")
except InvalidCandleError:
    pass

# --- Test 6: Validasi - close di luar [low, high] ditolak ---
try:
    Candle(date=datetime(2021, 1, 8), open=1.0, high=1.5,
           low=0.8, close=2.0, volume=100.0)
    raise AssertionError("Seharusnya raise InvalidCandleError")
except InvalidCandleError:
    pass

# --- Test 7: Immutability (frozen=True) ---
try:
    c_bull.open = 99.0  # type: ignore
    raise AssertionError("Frozen dataclass seharusnya tidak bisa di-mutate")
except Exception as e:
    # FrozenInstanceError adalah subclass dari AttributeError
    assert "frozen" in str(e).lower() or "can't set" in str(e).lower() \
        or isinstance(e, AttributeError)

# --- Test 8: Hashable (penting karena frozen=True) ---
candle_set = {c_bull, c_bear}
assert len(candle_set) == 2

# --- Test 9: __str__ mengandung tanggal + tag bullish/bearish ---
s_bull = str(c_bull)
s_bear = str(c_bear)
assert "2021-01-04" in s_bull and "BULL" in s_bull
assert "2021-01-05" in s_bear and "BEAR" in s_bear

print("=" * 60)
print("  All 9 Candle assertions PASSED")
print("=" * 60)
print(f"  Bullish sample: {c_bull}")
print(f"  Bearish sample: {c_bear}")
print(f"  Return pct      bull = {c_bull.daily_return_pct():.4f}%")
print(f"  Return pct      bear = {c_bear.daily_return_pct():.4f}%")
print(f"  Range size      bull = {c_bull.range_size():.5f}")
print(f"  Body size       bull = {c_bull.body_size():.5f}")

  All 9 Candle assertions PASSED
  Bullish sample: Candle(2021-01-04 [BULL] O=1.22340 H=1.23115 L=1.22240 C=1.22990)
  Bearish sample: Candle(2021-01-05 [BEAR] O=1.23000 H=1.23100 L=1.22000 C=1.22500)
  Return pct      bull = 0.5313%
  Return pct      bear = -0.4065%
  Range size      bull = 0.00875
  Body size       bull = 0.00650


### 7.2 Repository Layer — `CandleRepository`

Menerapkan **Repository Pattern**: memisahkan akses data dari business logic. Tanggung jawab:

1. **Load CSV** dengan `pandas.read_csv`.
2. **Parse tanggal** menjadi `datetime` Python.
3. **Konversi** setiap baris menjadi instance `Candle` (validasi otomatis via `__post_init__`).
4. **Sortir** by tanggal (ascending).
5. Sediakan **query methods**: `get_all`, `get_by_year`, `get_range`, `get_bullish`, `get_bearish`.

Manfaat: jika nanti sumber data diganti (mis. dari API atau SQLite), service & analyzer **tidak perlu diubah**.


In [11]:
# Sel 7.2.1 - CandleRepository (Repository Pattern - Materi 12)
# Materi 14: OOP + File handling (CSV)
# Materi 04: Enkapsulasi (_candles, _file_path protected)
# Materi 09: Magic method __len__

class CandleRepository:
    """Repository untuk data Candle XAU/USD (Gold) harian.

    Tanggung jawab (Single Responsibility - Materi 11):
      1. Load CSV daily dari disk (separator ';', format Eropa-numerik).
      2. Parse timestamp format 'YYYY.MM.DD HH:MM' jadi datetime.
      3. Validasi & konversi tiap baris menjadi instance Candle.
      4. Sortir by date (ascending).
      5. Sediakan query methods untuk dipakai layer di atasnya.

    Manfaat Repository Pattern: kalau sumber data diganti (mis. dari
    API atau SQLite), Service & Analyzer TIDAK perlu diubah.

    Dataset XAU dari Kaggle (novandraanugrah) sudah pre-filtered:
    hanya hari trading (Senin-Jumat), tidak ada missing values,
    tidak ada duplikat tanggal. Repository tetap defensive — query
    sebelum load akan raise RepositoryNotLoadedError.
    """

    _DATE_FORMAT = "%Y.%m.%d %H:%M"
    _CSV_SEPARATOR = ";"

    def __init__(self) -> None:
        self._candles: list[Candle] = []
        self._file_path: str | None = None

    # ---------- Loading ----------
    def load_csv(self, file_path: str) -> int:
        """Load CSV daily XAU/USD dan konversi ke list[Candle].

        Returns:
            int: jumlah Candle yang berhasil dimuat.

        Raises:
            FileNotFoundError: kalau file tidak ada.
            InvalidCandleError: kalau ada row dengan data tidak valid.
        """
        path = Path(file_path)
        if not path.exists():
            raise FileNotFoundError(f"File tidak ditemukan: {file_path}")

        # 1. Baca CSV (note: separator ';', bukan default ',')
        df = pd.read_csv(path, sep=self._CSV_SEPARATOR)
        logger.info(f"Loaded {len(df):,} baris dari {path.name}")

        # 2. Parse kolom Date (format YYYY.MM.DD HH:MM)
        df["timestamp"] = pd.to_datetime(df["Date"], format=self._DATE_FORMAT)

        # 3. Sortir by tanggal (defensive, walau data Kaggle sudah terurut)
        df = df.sort_values("timestamp").reset_index(drop=True)

        # 4. Konversi tiap baris menjadi Candle (dataclass akan validasi)
        candles: list[Candle] = []
        for _, row in df.iterrows():
            candles.append(
                Candle(
                    date=row["timestamp"].to_pydatetime(),
                    open=float(row["Open"]),
                    high=float(row["High"]),
                    low=float(row["Low"]),
                    close=float(row["Close"]),
                    volume=float(row["Volume"]),
                )
            )

        # 5. Simpan state internal
        self._candles = candles
        self._file_path = str(path)
        return len(candles)

    # ---------- Internal helper ----------
    def _require_loaded(self) -> None:
        if not self._candles:
            raise RepositoryNotLoadedError(
                "Repository belum dimuat. Panggil load_csv() dulu."
            )

    # ---------- Query methods ----------
    def get_all(self) -> list[Candle]:
        """Return semua candle (copy defensif)."""
        self._require_loaded()
        return list(self._candles)

    def get_by_year(self, year: int) -> list[Candle]:
        """Filter candle untuk tahun tertentu."""
        self._require_loaded()
        return [c for c in self._candles if c.date.year == year]

    def get_range(self, start: date, end: date) -> list[Candle]:
        """Filter candle dalam rentang [start, end] inklusif."""
        self._require_loaded()
        s = start.date() if isinstance(start, datetime) else start
        e = end.date()   if isinstance(end,   datetime) else end
        return [c for c in self._candles if s <= c.date.date() <= e]

    def get_bullish(self) -> list[Candle]:
        """Candle bullish saja (close > open)."""
        self._require_loaded()
        return [c for c in self._candles if c.is_bullish()]

    def get_bearish(self) -> list[Candle]:
        """Candle bearish saja (close <= open)."""
        self._require_loaded()
        return [c for c in self._candles if not c.is_bullish()]

    def date_range(self) -> tuple[datetime, datetime] | None:
        """Tanggal pertama dan terakhir data, atau None kalau kosong."""
        if not self._candles:
            return None
        return self._candles[0].date, self._candles[-1].date

    # ---------- Magic methods ----------
    def __len__(self) -> int:
        return len(self._candles)

    def __repr__(self) -> str:
        n = len(self._candles)
        src = Path(self._file_path).name if self._file_path else "<unloaded>"
        return f"CandleRepository(n={n}, source={src!r})"


print("Class CandleRepository berhasil didefinisikan (XAU/USD daily).")
print("  Methods: load_csv, get_all, get_by_year, get_range,")
print("           get_bullish, get_bearish, date_range, __len__")

Class CandleRepository berhasil didefinisikan (XAU/USD daily).
  Methods: load_csv, get_all, get_by_year, get_range,
           get_bullish, get_bearish, date_range, __len__


In [12]:
# Sel 7.2.2 - Inline tests untuk CandleRepository (8 assertion)

CSV_PATH = "data/XAU_1d_data.csv"

# --- Test 1: query sebelum load harus raise ---
repo_empty = CandleRepository()
try:
    repo_empty.get_all()
    raise AssertionError("Seharusnya raise RepositoryNotLoadedError")
except RepositoryNotLoadedError:
    pass

# --- Test 2: load_csv berhasil ---
repo = CandleRepository()
n_loaded = repo.load_csv(CSV_PATH)
assert n_loaded > 0

# --- Test 3: jumlah candle masuk akal untuk ~21 tahun daily ---
# 21 tahun * 252 trading days/tahun ~= 5.300 candle
assert 5000 <= n_loaded <= 6000, \
    f"Jumlah candle tidak masuk akal: {n_loaded} (harusnya 5000-6000)"

# --- Test 4: tidak ada weekend (dataset Kaggle sudah pre-filtered) ---
weekdays = {c.date.weekday() for c in repo.get_all()}
assert weekdays <= {0, 1, 2, 3, 4}, \
    f"Ada candle weekend di dataset: {weekdays}"

# --- Test 5: __len__ konsisten ---
assert len(repo) == n_loaded

# --- Test 6: semua candle terurut by date ---
all_candles = repo.get_all()
for i in range(1, len(all_candles)):
    assert all_candles[i - 1].date < all_candles[i].date

# --- Test 7: get_by_year(2020) return jumlah masuk akal ---
candles_2020 = repo.get_by_year(2020)
# 2020 ada 253 trading days di US market
assert 240 < len(candles_2020) < 270, \
    f"Jumlah candle 2020 tidak masuk akal: {len(candles_2020)}"
assert all(c.date.year == 2020 for c in candles_2020)

# --- Test 8: date_range periode masuk akal ---
dr = repo.date_range()
assert dr is not None
first_date, last_date = dr
assert first_date.year == 2004, f"Tahun pertama: {first_date.year}"
assert last_date.year >= 2024, f"Tahun terakhir: {last_date.year}"

# --- Test 9: bullish + bearish = total ---
n_bull = len(repo.get_bullish())
n_bear = len(repo.get_bearish())
assert n_bull + n_bear == n_loaded

print("=" * 65)
print(f"  All 9 CandleRepository assertions PASSED")
print("=" * 65)
print(f"  Total daily candles : {n_loaded:,}")
print(f"  Periode             : {first_date.date()} s/d {last_date.date()}")
print(f"  Bullish vs Bearish  : {n_bull:,} vs {n_bear:,} "
      f"(ratio bullish = {n_bull/n_loaded:.1%})")
print(f"  Daily candle 2020   : {len(candles_2020)}")
print(f"  Repo repr           : {repo!r}")
print(f"  Sample first        : {all_candles[0]}")
print(f"  Sample last         : {all_candles[-1]}")

FileNotFoundError: File tidak ditemukan: data/XAU_1d_data.csv

### 7.3 Analyzer Layer — `BaseAnalyzer` + 6 Subclass

Inti dari sistem analisis. Menggunakan **Strategy Pattern via ABC**: setiap analyzer mengimplementasikan strategi analisis yang berbeda, tetapi dengan interface yang sama (`analyze() -> dict`).

| Subclass | Fokus Analisis |
|---|---|
| `TrendAnalyzer` | Tren tahunan, total return, perubahan harga |
| `VolatilityAnalyzer` | Std dev returns, range harian rata-rata |
| `ReturnsAnalyzer` | Distribusi returns, hari terbaik & terburuk |
| `MovingAverageAnalyzer` | SMA 50 & 200, golden cross / death cross |
| `PriceDistributionAnalyzer` | Statistik deskriptif (mean, median, percentile) |
| `LiquiditySwingAnalyzer` | Deteksi swing high/low (terinspirasi LuxAlgo) |


In [ ]:
# Sel 7.3.1 - BaseAnalyzer (ABC) - Materi 08 Abstraksi
# Strategy Pattern (Materi 12): tiap subclass = strategi analisis berbeda
# tapi interface seragam (analyze() -> dict).

class BaseAnalyzer(ABC):
    """Abstract base class untuk semua analyzer.

    Subclass WAJIB mengimplementasi method analyze() yang return dict.
    Materi 8 (ABC), Materi 6 (Inheritance), Materi 7 (Polymorphism).
    """

    def __init__(self, data: list[Candle], name: str | None = None) -> None:
        if not data:
            raise AnalysisError(
                f"{self.__class__.__name__}: data candle tidak boleh kosong"
            )
        # Defensive: simpan sebagai tuple agar protected dari mutasi luar
        self._data: list[Candle] = list(data)
        self._name: str = name or self.__class__.__name__

    @abstractmethod
    def analyze(self) -> dict:
        """Jalankan analisis dan return hasilnya sebagai dict.
        Subclass WAJIB override method ini."""
        ...

    # ----- Method konkret yang diwariskan ke semua subclass -----
    def get_name(self) -> str:
        return self._name

    def get_summary(self) -> dict:
        """Ringkasan dasar yang sama untuk semua analyzer."""
        return {
            "analyzer": self._name,
            "n_candles": len(self._data),
            "first_date": self._data[0].date.strftime("%Y-%m-%d"),
            "last_date": self._data[-1].date.strftime("%Y-%m-%d"),
        }

    def __len__(self) -> int:
        return len(self._data)

    def __repr__(self) -> str:
        return f"{self._name}(n={len(self._data)})"


print("BaseAnalyzer (ABC) berhasil didefinisikan.")
print("  Subclass WAJIB override @abstractmethod analyze() -> dict")

BaseAnalyzer (ABC) berhasil didefinisikan.
  Subclass WAJIB override @abstractmethod analyze() -> dict


In [ ]:
# Sel 7.3.2 - TrendAnalyzer
# Materi 6 (Inheritance) + Materi 7 (Polymorphism)

class TrendAnalyzer(BaseAnalyzer):
    """Analisis tren harga: return per tahun & total return periode."""

    def total_return_pct(self) -> float:
        """Return total dari candle pertama ke terakhir, dalam %."""
        first_close = self._data[0].close
        last_close = self._data[-1].close
        return (last_close - first_close) / first_close * 100.0

    def by_year(self) -> dict[int, dict]:
        """Return per tahun: {year: {start_close, end_close, return_pct, n}}."""
        result: dict[int, dict] = {}
        # Group candle by year
        years_data: dict[int, list[Candle]] = {}
        for c in self._data:
            years_data.setdefault(c.date.year, []).append(c)

        for year, candles in sorted(years_data.items()):
            start = candles[0].close
            end = candles[-1].close
            result[year] = {
                "start_close": start,
                "end_close": end,
                "return_pct": (end - start) / start * 100.0,
                "n_candles": len(candles),
            }
        return result

    def analyze(self) -> dict:
        by_yr = self.by_year()
        # Tahun terbaik & terburuk berdasarkan return
        sorted_yrs = sorted(by_yr.items(), key=lambda kv: kv[1]["return_pct"])
        return {
            **self.get_summary(),
            "total_return_pct": self.total_return_pct(),
            "n_years": len(by_yr),
            "best_year": sorted_yrs[-1][0] if sorted_yrs else None,
            "best_year_return_pct": sorted_yrs[-1][1]["return_pct"] if sorted_yrs else None,
            "worst_year": sorted_yrs[0][0] if sorted_yrs else None,
            "worst_year_return_pct": sorted_yrs[0][1]["return_pct"] if sorted_yrs else None,
            "by_year": by_yr,
        }


print("TrendAnalyzer OK")

TrendAnalyzer OK


In [ ]:
# Sel 7.3.3 - VolatilityAnalyzer

class VolatilityAnalyzer(BaseAnalyzer):
    """Analisis volatilitas: std dev daily returns, range rata-rata."""

    def _daily_returns(self) -> list[float]:
        """Daily return close-to-close, %."""
        out = []
        prev_close = self._data[0].close
        for c in self._data[1:]:
            out.append((c.close - prev_close) / prev_close * 100.0)
            prev_close = c.close
        return out

    def std_returns_pct(self) -> float:
        """Standar deviasi daily returns (close-to-close), dalam %."""
        returns = self._daily_returns()
        if len(returns) < 2:
            raise AnalysisError("Butuh minimal 2 candle untuk std dev")
        s = pd.Series(returns)
        return float(s.std(ddof=1))

    def range_avg(self) -> float:
        """Range rata-rata candle = mean(high - low)."""
        return float(sum(c.range_size() for c in self._data) / len(self._data))

    def volatility_by_year(self) -> dict[int, float]:
        """Std dev daily returns per tahun."""
        result: dict[int, float] = {}
        years_data: dict[int, list[Candle]] = {}
        for c in self._data:
            years_data.setdefault(c.date.year, []).append(c)
        for year, candles in sorted(years_data.items()):
            if len(candles) < 2:
                continue
            sub_returns = []
            prev = candles[0].close
            for c in candles[1:]:
                sub_returns.append((c.close - prev) / prev * 100.0)
                prev = c.close
            result[year] = float(pd.Series(sub_returns).std(ddof=1))
        return result

    def analyze(self) -> dict:
        vol_by_year = self.volatility_by_year()
        return {
            **self.get_summary(),
            "std_daily_return_pct": self.std_returns_pct(),
            "range_avg": self.range_avg(),
            "volatility_by_year": vol_by_year,
            "most_volatile_year": (
                max(vol_by_year, key=vol_by_year.get) if vol_by_year else None
            ),
            "least_volatile_year": (
                min(vol_by_year, key=vol_by_year.get) if vol_by_year else None
            ),
        }


print("VolatilityAnalyzer OK")

VolatilityAnalyzer OK


In [ ]:
# Sel 7.3.4 - ReturnsAnalyzer

class ReturnsAnalyzer(BaseAnalyzer):
    """Analisis distribusi daily returns: best/worst day, statistik distribusi."""

    def _daily_returns_indexed(self) -> list[tuple[datetime, float]]:
        """[(date, return_pct), ...] close-to-close antar candle."""
        out = []
        prev_close = self._data[0].close
        for c in self._data[1:]:
            ret = (c.close - prev_close) / prev_close * 100.0
            out.append((c.date, ret))
            prev_close = c.close
        return out

    def best_day(self) -> tuple[datetime, float]:
        """Hari dengan return tertinggi."""
        returns = self._daily_returns_indexed()
        if not returns:
            raise AnalysisError("Butuh minimal 2 candle untuk best/worst day")
        return max(returns, key=lambda x: x[1])

    def worst_day(self) -> tuple[datetime, float]:
        """Hari dengan return terendah."""
        returns = self._daily_returns_indexed()
        if not returns:
            raise AnalysisError("Butuh minimal 2 candle untuk best/worst day")
        return min(returns, key=lambda x: x[1])

    def returns_distribution(self) -> dict:
        """Statistik distribusi: mean, median, std, skew, kurtosis."""
        returns = [r for _, r in self._daily_returns_indexed()]
        s = pd.Series(returns)
        return {
            "n_days": len(returns),
            "mean_pct": float(s.mean()),
            "median_pct": float(s.median()),
            "std_pct": float(s.std(ddof=1)),
            "skew": float(s.skew()),
            "kurtosis": float(s.kurtosis()),
            "min_pct": float(s.min()),
            "max_pct": float(s.max()),
        }

    def analyze(self) -> dict:
        best = self.best_day()
        worst = self.worst_day()
        return {
            **self.get_summary(),
            "distribution": self.returns_distribution(),
            "best_day": {"date": best[0].strftime("%Y-%m-%d"), "return_pct": best[1]},
            "worst_day": {"date": worst[0].strftime("%Y-%m-%d"), "return_pct": worst[1]},
        }


print("ReturnsAnalyzer OK")

ReturnsAnalyzer OK


In [ ]:
# Sel 7.3.5 - MovingAverageAnalyzer

class MovingAverageAnalyzer(BaseAnalyzer):
    """Analisis Simple Moving Average (SMA) + golden/death cross."""

    def sma(self, window: int) -> list[tuple[datetime, float]]:
        """SMA(window) atas Close. Return [(date, sma_value)]."""
        if window <= 0:
            raise AnalysisError(f"window harus > 0, dapat {window}")
        if window > len(self._data):
            return []
        out = []
        closes = [c.close for c in self._data]
        for i in range(window - 1, len(closes)):
            avg = sum(closes[i - window + 1 : i + 1]) / window
            out.append((self._data[i].date, avg))
        return out

    def _cross_dates(self, fast: int, slow: int, golden: bool) -> list[datetime]:
        """Tanggal saat SMA(fast) cross SMA(slow). golden=True -> fast cross di atas."""
        sma_fast = self.sma(fast)
        sma_slow = self.sma(slow)
        # Align ke index slow (lebih panjang lag-nya)
        offset = slow - fast  # index sma_fast[offset:] sejajar dgn sma_slow
        if offset < 0:
            raise AnalysisError("slow harus >= fast")
        fast_aligned = sma_fast[offset:]
        if len(fast_aligned) != len(sma_slow):
            # Safety check
            n = min(len(fast_aligned), len(sma_slow))
            fast_aligned, sma_slow = fast_aligned[:n], sma_slow[:n]
        crosses: list[datetime] = []
        for i in range(1, len(sma_slow)):
            prev_diff = fast_aligned[i - 1][1] - sma_slow[i - 1][1]
            curr_diff = fast_aligned[i][1] - sma_slow[i][1]
            if golden and prev_diff <= 0 and curr_diff > 0:
                crosses.append(sma_slow[i][0])
            elif (not golden) and prev_diff >= 0 and curr_diff < 0:
                crosses.append(sma_slow[i][0])
        return crosses

    def golden_cross_dates(self, fast: int = 50, slow: int = 200) -> list[datetime]:
        return self._cross_dates(fast, slow, golden=True)

    def death_cross_dates(self, fast: int = 50, slow: int = 200) -> list[datetime]:
        return self._cross_dates(fast, slow, golden=False)

    def analyze(self) -> dict:
        sma50 = self.sma(50)
        sma200 = self.sma(200)
        gc = self.golden_cross_dates()
        dc = self.death_cross_dates()
        return {
            **self.get_summary(),
            "sma50_last": sma50[-1][1] if sma50 else None,
            "sma200_last": sma200[-1][1] if sma200 else None,
            "last_close": self._data[-1].close,
            "golden_cross_count": len(gc),
            "death_cross_count": len(dc),
            "golden_cross_dates": [d.strftime("%Y-%m-%d") for d in gc],
            "death_cross_dates": [d.strftime("%Y-%m-%d") for d in dc],
        }


print("MovingAverageAnalyzer OK")

MovingAverageAnalyzer OK


In [ ]:
# Sel 7.3.6 - PriceDistributionAnalyzer

class PriceDistributionAnalyzer(BaseAnalyzer):
    """Statistik deskriptif distribusi harga close: mean, percentile, dll."""

    def describe(self) -> dict:
        """Statistik deskriptif untuk Close price."""
        closes = pd.Series([c.close for c in self._data])
        return {
            "mean": float(closes.mean()),
            "median": float(closes.median()),
            "std": float(closes.std(ddof=1)),
            "min": float(closes.min()),
            "max": float(closes.max()),
            "q25": float(closes.quantile(0.25)),
            "q75": float(closes.quantile(0.75)),
        }

    def percentiles(self, qs: list[float] | None = None) -> dict[str, float]:
        """Percentile pada Close pada quantile yang diminta."""
        qs = qs or [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
        closes = pd.Series([c.close for c in self._data])
        return {f"p{int(q*100)}": float(closes.quantile(q)) for q in qs}

    def analyze(self) -> dict:
        return {
            **self.get_summary(),
            "describe": self.describe(),
            "percentiles": self.percentiles(),
        }


print("PriceDistributionAnalyzer OK")

PriceDistributionAnalyzer OK


In [ ]:
# Sel 7.3.7 - LiquiditySwingAnalyzer
# Terinspirasi indikator "Liquidity Swing" oleh LuxAlgo (TradingView).
# https://www.tradingview.com/v/1S2VOnJP/
#
# Konsep: deteksi swing high/low pakai pivot lookback N bar.
#   swing high: high[i] > semua high di [i-N, i-1] DAN > semua high di [i+1, i+N]
#   swing low : low[i]  < semua low  di [i-N, i-1] DAN < semua low  di [i+1, i+N]
#
# Algoritma pivot detection adalah teori TA klasik (public domain).
# Implementasi di sini sepenuhnya mandiri.

class LiquiditySwingAnalyzer(BaseAnalyzer):
    """Deteksi swing high/low sebagai proxy area liquidity.

    Default lookback = 20 untuk TF daily, lebih selektif daripada
    setting 10 yang umum dipakai di scalping TF rendah.
    """

    DEFAULT_LOOKBACK = 20

    def __init__(
        self,
        data: list[Candle],
        lookback: int = DEFAULT_LOOKBACK,
        name: str | None = None,
    ) -> None:
        super().__init__(data, name)
        if lookback < 1:
            raise AnalysisError(f"lookback harus >= 1, dapat {lookback}")
        if lookback * 2 + 1 > len(self._data):
            raise AnalysisError(
                f"data terlalu sedikit untuk lookback={lookback} "
                f"(butuh minimal {lookback*2+1} candle, ada {len(self._data)})"
            )
        self._lookback = lookback

    def detect_swing_highs(self) -> list[tuple[datetime, float]]:
        """Return list (date, high) untuk semua swing high terkonfirmasi."""
        N = self._lookback
        d = self._data
        swings: list[tuple[datetime, float]] = []
        for i in range(N, len(d) - N):
            window_left = [d[j].high for j in range(i - N, i)]
            window_right = [d[j].high for j in range(i + 1, i + N + 1)]
            if d[i].high > max(window_left) and d[i].high > max(window_right):
                swings.append((d[i].date, d[i].high))
        return swings

    def detect_swing_lows(self) -> list[tuple[datetime, float]]:
        """Return list (date, low) untuk semua swing low terkonfirmasi."""
        N = self._lookback
        d = self._data
        swings: list[tuple[datetime, float]] = []
        for i in range(N, len(d) - N):
            window_left = [d[j].low for j in range(i - N, i)]
            window_right = [d[j].low for j in range(i + 1, i + N + 1)]
            if d[i].low < min(window_left) and d[i].low < min(window_right):
                swings.append((d[i].date, d[i].low))
        return swings

    def find_major_levels(self, top_n: int = 5) -> dict[str, list[tuple[datetime, float]]]:
        """Top-N resistance (swing high tertinggi) & support (swing low terendah).

        Pendekatan sederhana: ambil N swing high TERTINGGI sebagai resistance,
        N swing low TERENDAH sebagai support. Setiap swing point dianggap
        sebagai 'major level' independen.
        """
        highs = self.detect_swing_highs()
        lows = self.detect_swing_lows()
        resistance = sorted(highs, key=lambda x: x[1], reverse=True)[:top_n]
        support = sorted(lows, key=lambda x: x[1])[:top_n]
        return {"resistance": resistance, "support": support}

    def analyze(self) -> dict:
        highs = self.detect_swing_highs()
        lows = self.detect_swing_lows()
        levels = self.find_major_levels(top_n=5)
        return {
            **self.get_summary(),
            "lookback": self._lookback,
            "n_swing_highs": len(highs),
            "n_swing_lows": len(lows),
            "major_resistance": [
                {"date": d.strftime("%Y-%m-%d"), "price": p}
                for d, p in levels["resistance"]
            ],
            "major_support": [
                {"date": d.strftime("%Y-%m-%d"), "price": p}
                for d, p in levels["support"]
            ],
            "all_swing_highs": [(d.strftime("%Y-%m-%d"), p) for d, p in highs],
            "all_swing_lows": [(d.strftime("%Y-%m-%d"), p) for d, p in lows],
        }


print("LiquiditySwingAnalyzer OK (terinspirasi LuxAlgo Liquidity Swing)")

LiquiditySwingAnalyzer OK (terinspirasi LuxAlgo Liquidity Swing)


In [ ]:
# Sel 7.3.8 - Inline tests untuk semua Analyzer (13 assertion)
# Menggunakan repo dari sel 7.2.2 yang sudah loaded (XAU/USD daily)

# --- Test 1: BaseAnalyzer ABC - tidak bisa di-instantiate ---
try:
    BaseAnalyzer(repo.get_all())  # type: ignore
    raise AssertionError("ABC seharusnya tidak bisa di-instantiate")
except TypeError:
    pass

# --- Test 2: Empty data raise AnalysisError ---
try:
    TrendAnalyzer([])
    raise AssertionError("Empty data seharusnya raise AnalysisError")
except AnalysisError:
    pass

# --- Buat semua analyzer dari data XAU real ---
sample = repo.get_all()
trend = TrendAnalyzer(sample)
vol   = VolatilityAnalyzer(sample)
ret   = ReturnsAnalyzer(sample)
ma    = MovingAverageAnalyzer(sample)
pdist = PriceDistributionAnalyzer(sample)
liq   = LiquiditySwingAnalyzer(sample, lookback=20)

# --- Test 3: TrendAnalyzer.by_year() struktur benar ---
by_year = trend.by_year()
assert 2004 in by_year and 2020 in by_year
for y, info in by_year.items():
    assert {"start_close", "end_close", "return_pct", "n_candles"} <= info.keys()

# --- Test 4: total_return_pct positif besar (Gold reli 2004-2024) ---
total = trend.total_return_pct()
# XAU: $384 (2004) -> $4889+ (2026) = ~+1180%
assert total > 100, f"total_return seharusnya jauh > 100% (gold reli), dapat {total}%"

# --- Test 5: VolatilityAnalyzer.std_returns_pct masuk akal untuk gold ---
std_ret = vol.std_returns_pct()
assert std_ret > 0
# Gold daily volatility biasanya 1-2%
assert 0.5 < std_ret < 3.0, f"std daily return gold tidak realistis: {std_ret}%"

# --- Test 6: ReturnsAnalyzer best/worst ---
best_date, best_ret = ret.best_day()
worst_date, worst_ret = ret.worst_day()
assert best_ret > 0
assert worst_ret < 0
assert best_ret > worst_ret

# --- Test 7: MovingAverageAnalyzer.sma() length benar ---
sma_50 = ma.sma(50)
sma_200 = ma.sma(200)
assert len(sma_50) == len(sample) - 49
assert len(sma_200) == len(sample) - 199
# SMA value harus dalam range harga gold ($300-$6000)
assert all(300 < v < 6000 for _, v in sma_50)

# --- Test 8: Golden cross & death cross detected ---
gc_dates = ma.golden_cross_dates()
dc_dates = ma.death_cross_dates()
assert len(gc_dates) > 0, "Tidak ada golden cross terdeteksi"
assert len(dc_dates) > 0, "Tidak ada death cross terdeteksi"

# --- Test 9: PriceDistributionAnalyzer.describe() ---
desc = pdist.describe()
assert {"mean","median","std","min","max","q25","q75"} <= desc.keys()
assert desc["min"] < desc["q25"] < desc["median"] < desc["q75"] < desc["max"]
# Mean gold historis dataset ini ~$1400
assert 800 < desc["mean"] < 3000, f"mean close gold: {desc['mean']}"

# --- Test 10: LiquiditySwingAnalyzer detect ---
swings_hi = liq.detect_swing_highs()
swings_lo = liq.detect_swing_lows()
assert len(swings_hi) > 0
assert len(swings_lo) > 0
assert all(d < sample[-20].date for d, _ in swings_hi)

# --- Test 11: Major levels struktur benar ---
levels = liq.find_major_levels(top_n=5)
assert len(levels["resistance"]) <= 5
assert len(levels["support"]) <= 5
max_resistance = max(p for _, p in levels["resistance"])
min_support = min(p for _, p in levels["support"])
assert max_resistance > min_support

# --- Test 12: Polymorphism - semua analyzer punya analyze() yang return dict ---
analyzers = [trend, vol, ret, ma, pdist, liq]
for a in analyzers:
    result = a.analyze()
    assert isinstance(result, dict)
    assert "analyzer" in result and "n_candles" in result

# --- Test 13: LiquiditySwingAnalyzer lookback parameter effect ---
liq10 = LiquiditySwingAnalyzer(sample, lookback=10)
liq30 = LiquiditySwingAnalyzer(sample, lookback=30)
assert len(liq10.detect_swing_highs()) > len(liq30.detect_swing_highs())

print("=" * 65)
print(f"  All 13 Analyzer assertions PASSED (XAU/USD daily)")
print("=" * 65)
print(f"  Total return 2004-2026   : {total:+.2f}%   <-- gold reli besar")
print(f"  Std daily return         : {std_ret:.4f}%")
print(f"  Best day                 : {best_date.date()} ({best_ret:+.4f}%)")
print(f"  Worst day                : {worst_date.date()} ({worst_ret:+.4f}%)")
print(f"  Golden cross count (50/200): {len(gc_dates)}")
print(f"  Death cross count  (50/200): {len(dc_dates)}")
print(f"  Mean close price         : ${desc['mean']:.2f}")
print(f"  Median close price       : ${desc['median']:.2f}")
print(f"  Min - Max                : ${desc['min']:.2f} - ${desc['max']:.2f}")
print(f"  Swing highs (lookback=20): {len(swings_hi)}")
print(f"  Swing lows  (lookback=20): {len(swings_lo)}")
print(f"  Swing highs (lookback=10): {len(liq10.detect_swing_highs())}")
print(f"  Swing highs (lookback=30): {len(liq30.detect_swing_highs())}")

  All 13 Analyzer assertions PASSED (XAU/USD daily)
  Total return 2004-2026   : +1172.97%   <-- gold reli besar
  Std daily return         : 1.1133%
  Best day                 : 2025-10-15 (+15.5080%)
  Worst day                : 2026-01-30 (-9.0668%)
  Golden cross count (50/200): 15
  Death cross count  (50/200): 15
  Mean close price         : $1414.76
  Median close price       : $1301.84
  Min - Max                : $382.80 - $5414.46
  Swing highs (lookback=20): 89
  Swing lows  (lookback=20): 90
  Swing highs (lookback=10): 175
  Swing highs (lookback=30): 58


### 7.4 Service Layer — `ForexAnalysisService`

Lapisan paling atas dari hierarki OOP. Tanggung jawab:

- **Komposisi** semua analyzer (di-instansiasi sekali, di-reuse).
- **Orkestrasi** `run_full_analysis()` — panggil semua analyzer dan gabungkan hasilnya.
- **Ekspor** report ke JSON untuk dokumentasi.
- **Ringkasan tekstual** untuk display di notebook.

Service **tergantung pada abstraksi** (`BaseAnalyzer`), bukan implementasi konkret — penerapan Dependency Inversion Principle.


In [ ]:
# Sel 7.4.1 - ForexAnalysisService (Service Layer)
# Materi 12 (Design Pattern): Service yg mengorkestrasi Repository + Analyzers
# Materi 13 (SOLID-DIP): bergantung pada abstraksi (BaseAnalyzer), bukan
#   implementasi konkret. Analyzer baru bisa ditambah tanpa modifikasi
#   class ini (Open/Closed Principle).
# Komposisi (◆): Service "memiliki" list analyzer (lifecycle terikat ke Service).
# Agregasi (◇) : Service "memakai" Repository (lifecycle Repo independen).

class ForexAnalysisService:
    """Orkestrator analisis XAU/USD.

    Menggabungkan satu CandleRepository (sumber data) dengan banyak
    BaseAnalyzer (strategi analisis) — pola Strategy + Composition.

    Workflow:
        1. service = ForexAnalysisService(repo)
        2. service.register_analyzer(TrendAnalyzer(repo.get_all()))
        3. report = service.run_full_analysis()
        4. service.export_report_json("outputs/reports/full.json")

    Polymorphism (Materi 07): semua analyzer di-call lewat method
    .analyze() yang sama meski implementasinya berbeda-beda.
    """

    def __init__(self, repository: CandleRepository) -> None:
        if not isinstance(repository, CandleRepository):
            raise TypeError(
                f"repository harus CandleRepository, dapat {type(repository).__name__}"
            )
        self._repository: CandleRepository = repository
        self._analyzers: list[BaseAnalyzer] = []

    # ---------- Registrasi analyzer ----------
    def register_analyzer(self, analyzer: BaseAnalyzer) -> "ForexAnalysisService":
        """Tambahkan analyzer ke service. Return self untuk method chaining."""
        if not isinstance(analyzer, BaseAnalyzer):
            raise TypeError(
                f"analyzer harus subclass BaseAnalyzer, dapat {type(analyzer).__name__}"
            )
        self._analyzers.append(analyzer)
        logger.info(f"Registered: {analyzer.get_name()}")
        return self

    def get_analyzers(self) -> list[BaseAnalyzer]:
        """Salinan defensif list analyzer."""
        return list(self._analyzers)

    # ---------- Pipeline utama ----------
    def run_full_analysis(self) -> dict:
        """Jalankan .analyze() pada semua analyzer terdaftar.

        POLIMORFISME: tiap analyzer punya implementasi .analyze() berbeda,
        tapi dipanggil lewat interface yang sama (BaseAnalyzer.analyze).
        """
        if not self._analyzers:
            raise AnalysisError(
                "Belum ada analyzer yang diregister. "
                "Panggil register_analyzer() dulu."
            )

        dr = self._repository.date_range()
        report: dict = {
            "dataset": {
                "source": "XAU/USD daily (Kaggle - novandraanugrah)",
                "n_candles": len(self._repository),
                "first_date": dr[0].strftime("%Y-%m-%d") if dr else None,
                "last_date":  dr[1].strftime("%Y-%m-%d") if dr else None,
            },
            "n_analyzers": len(self._analyzers),
            "analyses": {},
        }

        for analyzer in self._analyzers:
            name = analyzer.get_name()
            try:
                report["analyses"][name] = analyzer.analyze()
            except Exception as exc:  # defensive: 1 analyzer gagal != pipeline mati
                logger.error(f"{name} gagal: {exc}")
                report["analyses"][name] = {"error": str(exc)}
        return report

    # ---------- Export & display ----------
    def export_report_json(self, file_path: str) -> str:
        """Tulis report hasil run_full_analysis() ke file JSON."""
        report = self.run_full_analysis()
        path = Path(file_path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            # default=str menangani datetime / objek non-serializable lain
            json.dump(report, f, indent=2, default=str)
        logger.info(f"Report tersimpan: {path}  ({path.stat().st_size:,} bytes)")
        return str(path)

    def summary_text(self) -> str:
        """Ringkasan teks human-readable (untuk di-print di notebook)."""
        report = self.run_full_analysis()
        ds = report["dataset"]
        lines = [
            "=" * 65,
            "  RINGKASAN ANALISIS XAU/USD (Gold)",
            "=" * 65,
            f"  Sumber data    : {ds['source']}",
            f"  Total candles  : {ds['n_candles']:,}",
            f"  Periode        : {ds['first_date']} s/d {ds['last_date']}",
            f"  Analyzer aktif : {report['n_analyzers']}",
            "-" * 65,
        ]
        for name, result in report["analyses"].items():
            lines.append(f"  [{name}]")
            if "error" in result:
                lines.append(f"      ERROR: {result['error']}")
                continue
            # Cetak hanya field skalar (skip dict bersarang spt by_year)
            for k, v in result.items():
                if isinstance(v, (int, float, str)) and k not in {"analyzer", "first_date", "last_date"}:
                    if isinstance(v, float):
                        lines.append(f"      {k:25s}: {v:.4f}")
                    else:
                        lines.append(f"      {k:25s}: {v}")
        lines.append("=" * 65)
        return "\n".join(lines)

    # ---------- Magic methods ----------
    def __len__(self) -> int:
        return len(self._analyzers)

    def __repr__(self) -> str:
        return (
            f"ForexAnalysisService(repo={self._repository!r}, "
            f"n_analyzers={len(self._analyzers)})"
        )


print("Class ForexAnalysisService berhasil didefinisikan.")
print("  Komposisi : list[BaseAnalyzer]  (◆)")
print("  Agregasi  : CandleRepository    (◇)")
print("  Methods   : register_analyzer, run_full_analysis,")
print("              export_report_json, summary_text")

Class ForexAnalysisService berhasil didefinisikan.
  Komposisi : list[BaseAnalyzer]  (◆)
  Agregasi  : CandleRepository    (◇)
  Methods   : register_analyzer, run_full_analysis,
              export_report_json, summary_text


In [ ]:
# Sel 7.4.2 - Inline tests untuk ForexAnalysisService (6 assertion)
# Reuse repo (sel 7.2.2) + analyzer instances (sel 7.3.8) yang sudah dibuat.

# --- Test 1: __init__ wajib terima CandleRepository ---
try:
    ForexAnalysisService("bukan repo")  # type: ignore
    raise AssertionError("Seharusnya raise TypeError")
except TypeError:
    pass

# --- Test 2: run_full_analysis tanpa analyzer harus raise ---
service_kosong = ForexAnalysisService(repo)
try:
    service_kosong.run_full_analysis()
    raise AssertionError("Seharusnya raise AnalysisError")
except AnalysisError:
    pass

# --- Test 3: register_analyzer hanya terima BaseAnalyzer ---
try:
    service_kosong.register_analyzer("bukan analyzer")  # type: ignore
    raise AssertionError("Seharusnya raise TypeError")
except TypeError:
    pass

# --- Bangun service utuh untuk test berikutnya ---
service = ForexAnalysisService(repo)
service.register_analyzer(trend) \
       .register_analyzer(vol)   \
       .register_analyzer(ret)   \
       .register_analyzer(ma)    \
       .register_analyzer(pdist) \
       .register_analyzer(liq)

assert len(service) == 6, f"Expected 6 analyzers, got {len(service)}"

# --- Test 4: run_full_analysis struktur output ---
report = service.run_full_analysis()
assert "dataset" in report and "analyses" in report and "n_analyzers" in report
assert report["n_analyzers"] == 6
assert report["dataset"]["n_candles"] == len(repo)

# --- Test 5: Polymorphism — semua 6 analyzer ter-execute ---
expected_names = {
    "TrendAnalyzer", "VolatilityAnalyzer", "ReturnsAnalyzer",
    "MovingAverageAnalyzer", "PriceDistributionAnalyzer",
    "LiquiditySwingAnalyzer",
}
assert set(report["analyses"].keys()) == expected_names, \
    f"Missing analyzers: {expected_names - set(report['analyses'].keys())}"
for name, result in report["analyses"].items():
    assert "error" not in result, f"{name} gagal: {result.get('error')}"
    assert "n_candles" in result and result["n_candles"] == len(repo)

# --- Test 6: export_report_json menulis file JSON valid ---
report_path = service.export_report_json("outputs/reports/full_report.json")
assert Path(report_path).exists()
assert Path(report_path).stat().st_size > 1000  # report kaya, bukan stub
# Pastikan JSON-nya valid (bisa di-load balik)
with open(report_path, encoding="utf-8") as f:
    reloaded = json.load(f)
assert reloaded["n_analyzers"] == 6

print("=" * 65)
print("  All 6 ForexAnalysisService assertions PASSED")
print("=" * 65)
print(f"  Service repr     : {service!r}")
print(f"  N analyzers      : {len(service)}")
print(f"  Report exported  : {report_path}")
print(f"  Report file size : {Path(report_path).stat().st_size:,} bytes")
print(f"  Analyzer names   : {sorted(report['analyses'].keys())}")
print()
print(service.summary_text())

[INFO] Registered: TrendAnalyzer


[INFO] Registered: VolatilityAnalyzer


[INFO] Registered: ReturnsAnalyzer


[INFO] Registered: MovingAverageAnalyzer


[INFO] Registered: PriceDistributionAnalyzer


[INFO] Registered: LiquiditySwingAnalyzer


[INFO] Report tersimpan: outputs/reports/full_report.json  (20,317 bytes)


  All 6 ForexAnalysisService assertions PASSED
  Service repr     : ForexAnalysisService(repo=CandleRepository(n=5531, source='XAU_1d_data.csv'), n_analyzers=6)
  N analyzers      : 6
  Report exported  : outputs/reports/full_report.json
  Report file size : 20,317 bytes
  Analyzer names   : ['LiquiditySwingAnalyzer', 'MovingAverageAnalyzer', 'PriceDistributionAnalyzer', 'ReturnsAnalyzer', 'TrendAnalyzer', 'VolatilityAnalyzer']

  RINGKASAN ANALISIS XAU/USD (Gold)
  Sumber data    : XAU/USD daily (Kaggle - novandraanugrah)
  Total candles  : 5,531
  Periode        : 2004-06-11 s/d 2026-01-30
  Analyzer aktif : 6
-----------------------------------------------------------------
  [TrendAnalyzer]
      n_candles                : 5531
      total_return_pct         : 1172.9706
      n_years                  : 23
      best_year                : 2025
      best_year_return_pct     : 62.4677
      worst_year               : 2013
      worst_year_return_pct    : -28.3616
  [VolatilityAnalyze

## 8. Eksekusi Pipeline Analisis

Pada bagian ini dijalankan pipeline lengkap dengan data sebenarnya: load CSV → resample → analisis → report.


In [ ]:
# Sel 8.1 - Load dataset
# TODO (Fase F9): repo = CandleRepository();
#   n = repo.load_csv('data/EURUSD_ForexTrading_4hrs.csv')
#   print('Loaded', n, 'daily candles')
pass

In [ ]:
# Sel 8.2 - Jalankan ForexAnalysisService
# TODO (Fase F9): service = ForexAnalysisService(repo)
#   result = service.run_full_analysis()
pass

In [ ]:
# Sel 8.3 - Display ringkasan
# TODO (Fase F9): print(service.summary_text())
pass

## 9. Hasil Analisis Numerik

Tabel-tabel hasil analisis ditampilkan di bawah ini. Angka-angka inilah yang nanti dimasukkan ke slide PPT.


In [ ]:
# Sel 9.1 - Statistik deskriptif harga & volume
# TODO (Fase F9): pakai pandas describe() dari hasil PriceDistributionAnalyzer
pass

In [ ]:
# Sel 9.2 - Return per tahun
# TODO (Fase F9): tabel dari TrendAnalyzer.by_year()
pass

In [ ]:
# Sel 9.3 - Top 5 major support & resistance levels (LiquiditySwingAnalyzer)
# TODO (Fase F9): tampilkan level kunci dari find_major_levels(top_n=5)
pass

In [ ]:
# Sel 9.4 - Export report ke JSON
# TODO (Fase F9): service.export_report_json('outputs/reports/full_analysis.json')
pass

## 10. Visualisasi & Interpretasi

Lima grafik utama untuk slide presentasi. Semua disimpan ke `outputs/charts/*.png`.


In [ ]:
# Sel 10.1 - Grafik 1: Line chart EUR/USD Close Price (2003-2021)
# TODO (Fase F10): plt.plot(...) + simpan ke outputs/charts/01_close_price.png
pass

In [ ]:
# Sel 10.2 - Grafik 2: Histogram daily returns
# TODO (Fase F10): histogram + simpan
pass

In [ ]:
# Sel 10.3 - Grafik 3: Box plot volatility per tahun
# TODO (Fase F10): boxplot std dev returns per tahun + simpan
pass

In [ ]:
# Sel 10.4 - Grafik 4: Close + SMA 50 + SMA 200 + crossover markers
# TODO (Fase F10): line chart kombinasi + simpan
pass

In [ ]:
# Sel 10.5 - Grafik 5: Close + Swing High/Low markers + major levels
# (LiquiditySwingAnalyzer)
# TODO (Fase F10): scatter v/^ overlay + garis horizontal level + simpan
pass

## 11. Pembahasan

> *(Akan diisi setelah eksekusi penuh, dengan angka real dari hasil analyzer.)*

### Pengalaman Trading Penulis dan Relevansinya

Penulis memiliki pengalaman trading pribadi pada pasangan **XAU/USD** menggunakan indikator **Liquidity Swing oleh LuxAlgo** di TradingView pada timeframe 1 menit dengan pivot lookback 10. Indikator ini membantu mengidentifikasi area likuiditas — zona harga di mana stop loss banyak terkumpul — yang sering menjadi titik balik signifikan.

Pada proyek ini, konsep tersebut **diadaptasi ke timeframe daily** dengan **pivot lookback 20** (lebih selektif untuk menangkap swing yang signifikan dalam skala minggu/bulan). Implementasi dilakukan secara mandiri dalam class `LiquiditySwingAnalyzer`, mengikuti arsitektur OOP proyek. Hal ini menunjukkan bagaimana pengetahuan praktis dari trading retail dapat diformalisasi dalam framework yang terstruktur dan dapat diuji.

### Diskusi Hasil Analisis

1. **Karakteristik tren jangka panjang** — bagaimana XAU/USD bergerak selama ~21 tahun? Periode bull run 2008–2011 (krisis), correction 2013–2018, dan reli pasca-COVID 2020+.
2. **Periode volatilitas tinggi vs rendah** — krisis 2008, COVID 2020, perang Ukraina & inflasi 2022.
3. **Distribusi returns** — emas cenderung memiliki *positive skew* (lonjakan ke atas lebih ekstrem dari penurunan).
4. **Efektivitas Moving Average** — Golden cross & death cross historis emas, apakah memberi sinyal akurat?
5. **Level support/resistance kunci** — dari `LiquiditySwingAnalyzer`, level mana yang berulang kali diuji sepanjang sejarah?
6. **Penerapan OOP** — bagaimana arsitektur ini memudahkan pengembangan & maintenance? Bagaimana penambahan analyzer baru dapat dilakukan tanpa mengubah kode existing (Open-Closed Principle)?
7. **Keterbatasan** — granularitas daily (bukan tick), tidak ada data fundamental (suku bunga, CPI, geopolitik), dan dataset terbatas pada periode 2004–2024.


## 12. Kesimpulan & Saran

### Kesimpulan

> *(Akan diisi dengan poin-poin kuantitatif setelah analisis selesai.)*

### Saran Pengembangan

- Tambahkan **model prediksi** (Linear Regression, ARIMA, atau LSTM) untuk forecasting harga emas.
- Implementasi **backtesting framework** untuk uji strategi trading berbasis swing point pada data historis.
- Perluas ke **multi-asset** (XAG/USD perak, pasangan mata uang, indeks saham, kripto) — arsitektur OOP sudah siap dengan menambah subclass `CandleRepository` per aset.
- Buat **dashboard interaktif** dengan Streamlit atau Dash untuk visualisasi real-time.
- Tambah **data fundamental** (suku bunga Fed, CPI AS, DXY, event geopolitik) sebagai variabel pelengkap untuk konteks analisis.
- Bandingkan **TF Daily vs TF 1H atau 4H** dari dataset yang sama untuk melihat konsistensi swing point antar timeframe.


## 13. Daftar Pustaka

1. Anugrah, N. (2024). *XAU/USD Gold Price Historical Data 2004–2024* [Dataset]. Kaggle. https://www.kaggle.com/datasets/novandraanugrah/xauusd-gold-price-historical-data-2004-2024
2. Python Software Foundation. (2026). *Python 3 Documentation*. https://docs.python.org/3/
3. McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly Media.
4. pandas Development Team. (2026). *pandas Documentation*. https://pandas.pydata.org/docs/
5. Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering*, 9(3), 90–95.
6. Gamma, E., Helm, R., Johnson, R., & Vlissides, J. (1994). *Design Patterns: Elements of Reusable Object-Oriented Software*. Addison-Wesley.
7. Martin, R. C. (2008). *Clean Code: A Handbook of Agile Software Craftsmanship*. Prentice Hall.
8. Pramono, A. (2026). *Pemrograman Berorientasi Objek Menggunakan Python*. Universitas Mulawarman.
9. LuxAlgo. (n.d.). *Liquidity Swings* [TradingView indicator]. https://www.tradingview.com/v/1S2VOnJP/ — sumber inspirasi konsep `LiquiditySwingAnalyzer`, digunakan penulis dalam trading XAU/USD pada timeframe 1 menit; algoritma pivot detection diimplementasikan ulang secara mandiri pada timeframe daily.
10. Murphy, J. J. (1999). *Technical Analysis of the Financial Markets*. New York Institute of Finance.
11. World Gold Council. (2024). *Gold Market Outlook*. https://www.gold.org/


---

**TERIMA KASIH**

*Pertanyaan & Diskusi dipersilakan.*

> *"Code bukan hanya tentang mesin yang mengerti, tapi tentang manusia yang bisa membacanya."*

Proyek Akhir PBO 2026 | Informatika | Universitas Mulawarman
